# Tutorial 12 — Physics-Informed Neural Networks on a Disk and a Sphere

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 12: Physics-Informed Neural Networks for Geometry I**

---

A PINN uses a neural network as the **trial function** of a PDE solver. Nothing is
learned from data: the network $u_\theta$ is trained so that the equation holds at
sampled **collocation points**, with automatic differentiation supplying the
derivatives the equation needs (Lecture 4's autodiff, now differentiating *outputs
with respect to inputs*).

This tutorial does what Lecture 12's summary promises — **Poisson on a disk and on a
surface, compared with classical baselines** — and it keeps the lecture's promise to be
honest about the comparison. Every problem has a manufactured exact solution, so every
error below is a true error, not a training loss.

| § | Question | Lecture 12 |
|---|---|---|
| 1 | How does a PINN solve Poisson on the disk, and how accurately? | slides 4, 5, 7 |
| 2 | Hard vs soft boundary conditions — and a finite-element baseline | slides 3, 6, 11 |
| 3 | A PDE on a surface: the Laplace–Beltrami operator by autodiff | slide 8 |
| 4 | Deep Ritz: minimising an energy instead of a residual | slide 9 |
| 5 | What to take away | |

You need the `aigeo` environment from [Tutorial 1](../tutorial_01/README.md).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.spatial import Delaunay, ConvexHull
from scipy.interpolate import LinearNDInterpolator

torch.set_default_dtype(torch.float64)          # PINNs chasing 1e-5 accuracy want float64
grad = torch.autograd.grad

SEED = 20261012
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("torch", torch.__version__)

---
## 1. A PINN for Poisson on the disk

The lecture's worked example (slide 7). On the unit disk $\mathbb D$,

$$-\Delta u = 4 + 8x_1 \quad\text{in } \mathbb D, \qquad u = 0 \quad \text{on } \partial\mathbb D,$$

whose exact solution is $u^\ast(x) = (1-\lVert x\rVert^2)(1+x_1)$ — a *manufactured*
solution: we chose $u^\ast$ first and computed the forcing from it, so that every error
we report is a real error.

Three ingredients, each from a slide:

- **Autodiff builds the residual** (slide 5). Differentiating the network's output with
  respect to its *input* twice gives $\Delta u_\theta$ exactly — no finite-difference
  stencil. We check that on $u^\ast$ itself before trusting it on a network.
- **A smooth network.** $\tanh$ activations make $u_\theta$ infinitely differentiable. A
  ReLU network is piecewise linear, so its second derivatives vanish almost everywhere and
  the residual would be meaningless (Exercise 1b).
- **A hard boundary condition** (slide 6). Writing $u_\theta = (1 - \lVert x\rVert^2)\,N_\theta(x)$
  makes $u_\theta = 0$ on the circle *for every* $\theta$, so the loss needs only the PDE
  residual.

In [ ]:
def u_exact(x):
    return (1 - (x**2).sum(-1)) * (1 + x[..., 0])


def forcing(x):
    return 4 + 8 * x[..., 0]


def disk_points(n, rng):
    '''Uniform random points in the unit disk.'''
    r = np.sqrt(rng.uniform(size=n))
    t = rng.uniform(0, 2 * np.pi, n)
    return np.stack([r * np.cos(t), r * np.sin(t)], axis=1)


def laplacian(u, x):
    '''Laplacian of u(x) with respect to the input x, by autodiff.'''
    g = grad(u.sum(), x, create_graph=True)[0]
    return sum(grad(g[:, k].sum(), x, create_graph=True)[0][:, k] for k in range(x.shape[1]))


x_chk = torch.tensor(disk_points(1000, rng), requires_grad=True)
err = (-laplacian(u_exact(x_chk), x_chk) - forcing(x_chk)).abs().max().item()
print(f"autodiff check on the exact solution:  max | -Lap u* - f | = {err:.1e}")

Exact to floating point, as slide 5 said. The errors in a PINN come from the *network*
(approximation) and the *points* (sampling) — never from the derivatives.

Now the training loop. It is Lecture 3's canonical loop with two refinements from slide
7: **fresh collocation points every step** (so the network cannot overfit a fixed set),
then a short **L-BFGS** finish on a fixed set — a quasi-Newton method that uses curvature
information and is very effective once Adam has found the right basin.

In [ ]:
def mlp(d_in, width=50, depth=3):
    layers, d = [], d_in
    for _ in range(depth):
        layers += [torch.nn.Linear(d, width), torch.nn.Tanh()]
        d = width
    return torch.nn.Sequential(*layers, torch.nn.Linear(d, 1))


def train(loss_fn, params, sample, fixed, adam_steps=1000, lbfgs_iters=300,
          lr=2e-3, seed=0, evaluate=None):
    '''Adam on freshly sampled points, then L-BFGS on a fixed set.'''
    r = np.random.default_rng(seed)
    as_t = lambda arrays: [torch.tensor(a, requires_grad=True) for a in arrays]
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, adam_steps)
    hist, t0 = [], time.time()
    for step in range(adam_steps):
        loss = loss_fn(*as_t(sample(r)))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        hist.append(loss.item())
    out = {"t_adam": time.time() - t0, "n_adam": adam_steps}
    if evaluate is not None:
        out["err_adam"] = evaluate()

    x_fixed = as_t(fixed)
    lbfgs = torch.optim.LBFGS(params, max_iter=lbfgs_iters, line_search_fn="strong_wolfe")
    def closure():
        lbfgs.zero_grad()
        loss = loss_fn(*x_fixed)
        loss.backward()
        hist.append(loss.item())
        return loss
    t1 = time.time()
    lbfgs.step(closure)
    out["t_lbfgs"] = time.time() - t1
    if evaluate is not None:
        out["err"] = evaluate()
    out["hist"] = np.array(hist)
    return out


X_TEST = torch.tensor(disk_points(20000, np.random.default_rng(1)))       # independent test points
U_TEST = u_exact(X_TEST)


def rel_l2(u_fn):
    with torch.no_grad():
        return (torch.linalg.norm(u_fn(X_TEST) - U_TEST) / torch.linalg.norm(U_TEST)).item()


torch.manual_seed(0)
net_hard = mlp(2)
u_hard = lambda x: (1 - (x**2).sum(1)) * net_hard(x).squeeze(1)       # zero on the circle, always

def loss_residual(x):
    return ((-laplacian(u_hard(x), x) - forcing(x))**2).mean()

res_hard = train(loss_residual, list(net_hard.parameters()),
                 sample=lambda r: (disk_points(500, r),),
                 fixed=(disk_points(4000, np.random.default_rng(7)),),
                 evaluate=lambda: rel_l2(u_hard))

print(f"after {res_hard['n_adam']} Adam steps ({res_hard['t_adam']:.1f}s):  relative L2 error {res_hard['err_adam']:.2e}")
print(f"after L-BFGS ({res_hard['t_lbfgs']:.1f}s):             relative L2 error {res_hard['err']:.2e}")

In [ ]:
g = np.linspace(-1, 1, 201)
GX, GY = np.meshgrid(g, g)
inside = GX**2 + GY**2 <= 1
G = torch.tensor(np.stack([GX.ravel(), GY.ravel()], axis=1))
with torch.no_grad():
    U_star = u_exact(G).numpy().reshape(GX.shape)
    U_pinn = u_hard(G).numpy().reshape(GX.shape)
U_star[~inside] = np.nan
E_pinn = np.where(inside, np.abs(U_pinn - U_star), np.nan)

fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.4))
im = axes[0].imshow(U_star, extent=[-1, 1, -1, 1], origin="lower", cmap="viridis")
axes[0].set_title("exact solution $u^*$"); fig.colorbar(im, ax=axes[0], shrink=0.8)
im = axes[1].imshow(np.log10(E_pinn + 1e-12), extent=[-1, 1, -1, 1], origin="lower", cmap="magma", vmin=-7)
axes[1].set_title(r"PINN error, $\log_{10}|u_\theta - u^*|$"); fig.colorbar(im, ax=axes[1], shrink=0.8)
for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
h = res_hard["hist"]
axes[2].semilogy(np.arange(len(h)), h, color=GEO_DARK, lw=0.8)
axes[2].axvline(res_hard["n_adam"], color=GEO_RUST, ls="--", lw=1.2, label="Adam  |  L-BFGS")
axes[2].set_xlabel("step (Adam) / function evaluation (L-BFGS)"); axes[2].set_ylabel("residual loss")
axes[2].set_title("training"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

The PINN reaches a relative $L^2$ error of order $10^{-5}$, and the short L-BFGS finish
buys roughly an order of magnitude over Adam alone. The error map shows *where* it is
worst: typically in the interior, away from the boundary, where the hard constraint
pins the solution down for free.

Note what we measured it against: **independent test points and the exact solution**,
not the training loss. Slide 11's warning — a low residual does not certify a correct
solution — is the reason. Here we can afford the check; in a real problem, a trusted
baseline plays the role of $u^\ast$.

> **Exercise 1 — the recipe, one piece at a time.**
> (a) Remove the L-BFGS finish and double the Adam steps instead. Compare error per
> second of wall-clock.
>
> (b) Replace `Tanh` by `ReLU`. Before running it, predict what the residual loss will do
> and what the solution will look like. Then check, and relate it to slide 5.
>
> (c) **Adaptive sampling** (slide 5). Every 200 steps, evaluate the residual on 5000
> random points and add the 250 worst to the collocation set. Does it help here, where the
> solution is smooth everywhere? Construct a forcing term with a sharp feature where it
> should.

---
## 2. Soft boundary conditions, and a classical baseline

### Soft vs hard

The hard constraint used a special fact: the disk has the simple vanishing function
$1 - \lVert x\rVert^2$. The general-purpose alternative (slide 6) is a **soft** penalty:
let $u_\theta = N_\theta$ be unconstrained and add a boundary term to the loss,

$$L = L_{\text{res}} + \lambda_b\,L_{\text{bc}}, \qquad
L_{\text{bc}} = \frac1{N_b}\sum_j u_\theta(y_j)^2, \quad y_j \in \partial\mathbb D .$$

Same network, same recipe, two values of $\lambda_b$.

In [ ]:
def circle_points(n, rng):
    t = rng.uniform(0, 2 * np.pi, n)
    return np.stack([np.cos(t), np.sin(t)], axis=1)


T_CIRCLE = np.linspace(0, 2 * np.pi, 2000)
X_CIRCLE = torch.tensor(np.stack([np.cos(T_CIRCLE), np.sin(T_CIRCLE)], axis=1))

soft = {}
for lam_b in (1.0, 100.0):
    torch.manual_seed(0)
    net = mlp(2)
    u_soft = lambda x, net=net: net(x).squeeze(1)

    def loss_soft(xc, xb, u=u_soft, lam=lam_b):
        return ((-laplacian(u(xc), xc) - forcing(xc))**2).mean() + lam * (u(xb)**2).mean()

    out = train(loss_soft, list(net.parameters()),
                sample=lambda r: (disk_points(500, r), circle_points(100, r)),
                fixed=(disk_points(4000, np.random.default_rng(7)), circle_points(400, np.random.default_rng(8))),
                evaluate=lambda u=u_soft: rel_l2(u))
    xr = torch.tensor(disk_points(4000, np.random.default_rng(9)), requires_grad=True)
    out["residual"] = ((-laplacian(u_soft(xr), xr) - forcing(xr))**2).mean().item()
    with torch.no_grad():
        out["boundary"] = u_soft(X_CIRCLE).numpy()
    out["u"] = u_soft
    soft[lam_b] = out

xr = torch.tensor(disk_points(4000, np.random.default_rng(9)), requires_grad=True)
res_hard["residual"] = ((-laplacian(u_hard(xr), xr) - forcing(xr))**2).mean().item()
with torch.no_grad():
    res_hard["boundary"] = u_hard(X_CIRCLE).numpy()

print(f"{'':22s} {'rel. L2 error':>14s} {'max |u| on circle':>18s} {'residual loss':>14s}")
print(f"{'hard constraint':22s} {res_hard['err']:14.2e} {np.abs(res_hard['boundary']).max():18.1e} {res_hard['residual']:14.2e}")
for lam_b, out in soft.items():
    print(f"{f'soft, lambda_b = {lam_b:g}':22s} {out['err']:14.2e} {np.abs(out['boundary']).max():18.1e} {out['residual']:14.2e}")

The hard constraint is exact on the boundary by construction, and it is also the most
accurate *in the interior*: every unit of network capacity goes to the PDE. With a soft
penalty the two losses compete for the same parameters — slide 11's **loss imbalance** —
and raising $\lambda_b$ does not simply fix it: it buys boundary accuracy by giving up
residual accuracy. The printed residual column shows the trade. Use hard constraints when
you can build them, and report the boundary error separately when you cannot.

### The classical baseline

Slide 7's last bullet: *compare accuracy and runtime against a classical numerical
solve*. The natural baseline on a disk is the **finite element method** with piecewise
linear (P1) elements — about thirty lines with `scipy.sparse`:

1. triangulate points filling the disk (here a sunflower lattice plus boundary points);
2. assemble the **stiffness matrix** $K_{ij} = \int\nabla\varphi_i\cdot\nabla\varphi_j$ and
   the **mass matrix** $M_{ij} = \int\varphi_i\varphi_j$ of the hat functions $\varphi_i$;
3. solve $K u = M f$ on the interior nodes, with $u = 0$ on the boundary.

For a triangle with edge vectors $e_a$ opposite vertex $a$ and area $A$, the local
stiffness is $K_{ab} = e_a\cdot e_b/(4A)$. The same formula will work on a surface in §3.

In [ ]:
def p1_matrices(pts, tri):
    '''P1 stiffness and mass matrices on a triangle mesh in R^2 or R^3.'''
    P = pts[tri]
    e = np.stack([P[:, 2] - P[:, 1], P[:, 0] - P[:, 2], P[:, 1] - P[:, 0]], axis=1)   # edge opposite vertex a
    if pts.shape[1] == 2:
        A = 0.5 * np.abs(e[:, 1, 0] * e[:, 2, 1] - e[:, 1, 1] * e[:, 2, 0])
    else:
        A = 0.5 * np.linalg.norm(np.cross(e[:, 1], e[:, 2]), axis=1)
    K_loc = (e @ e.transpose(0, 2, 1)) / (4 * A[:, None, None])
    M_loc = (A / 12)[:, None, None] * (np.ones((3, 3)) + np.eye(3))
    rows, cols, n = np.repeat(tri, 3, axis=1).ravel(), np.tile(tri, (1, 3)).ravel(), len(pts)
    K = sp.coo_matrix((K_loc.ravel(), (rows, cols)), shape=(n, n)).tocsr()
    M = sp.coo_matrix((M_loc.ravel(), (rows, cols)), shape=(n, n)).tocsr()
    return K, M


def disk_mesh(n_interior):
    h = np.sqrt(np.pi / n_interior)
    n_bnd = int(round(2 * np.pi / h))
    j = np.arange(n_interior)
    r = (1 - 0.5 * h) * np.sqrt((j + 0.5) / n_interior)
    t = j * np.pi * (3 - np.sqrt(5))
    tb = 2 * np.pi * np.arange(n_bnd) / n_bnd
    pts = np.vstack([np.stack([r * np.cos(t), r * np.sin(t)], 1), np.stack([np.cos(tb), np.sin(tb)], 1)])
    return pts, np.arange(n_interior, n_interior + n_bnd)


def fem_disk(n_interior):
    pts, bnd = disk_mesh(n_interior)
    mesh = Delaunay(pts)
    K, M = p1_matrices(pts, mesh.simplices)
    b = M @ forcing(pts)
    free = np.setdiff1d(np.arange(len(pts)), bnd)
    u_h = np.zeros(len(pts))
    u_h[free] = spla.spsolve(K[free][:, free].tocsc(), b[free])
    return pts, mesh, u_h


def best_time(fn, *args, repeats=3):
    '''Run fn a few times; return its result and the fastest wall-clock time.'''
    times = []
    for _ in range(repeats):
        t0 = time.time()
        out = fn(*args)
        times.append(time.time() - t0)
    return out, min(times)


fem = []
print(f"{'nodes':>7s} {'rel. L2 error':>14s} {'time':>10s}")
for n_int in (100, 400, 1600, 6400, 25600, 51200):
    (pts, mesh, u_h), t_fem = best_time(fem_disk, n_int)
    u_at_test = LinearNDInterpolator(mesh, u_h, fill_value=0.0)(X_TEST.numpy())
    e = np.linalg.norm(u_at_test - U_TEST.numpy()) / np.linalg.norm(U_TEST.numpy())
    fem.append((len(pts), e, t_fem))
    print(f"{len(pts):7d} {e:14.2e} {t_fem * 1000:8.1f}ms")

In [ ]:
fem = np.array(fem)
fig, axes = plt.subplots(1, 2, figsize=(10.6, 3.6))

ax = axes[0]
ax.loglog(fem[:, 2], fem[:, 1], "o-", color=GEO_DARK, label="P1 finite elements")
for nodes, e, t in fem[::2]:
    ax.annotate(f"{int(nodes)}", (t, e), textcoords="offset points", xytext=(4, 4), fontsize=6.5, color=GEO_DARK)
ax.loglog(res_hard["t_adam"], res_hard["err_adam"], "s", color=GEO_TEAL, ms=7, label="PINN, Adam only")
ax.loglog(res_hard["t_adam"] + res_hard["t_lbfgs"], res_hard["err"], "*", color=GEO_TEAL, ms=12,
          label="PINN, + L-BFGS (hard BC)")
for lam_b, mk in [(1.0, "^"), (100.0, "v")]:
    o = soft[lam_b]
    ax.loglog(o["t_adam"] + o["t_lbfgs"], o["err"], mk, color=GEO_RUST, ms=8, label=f"PINN, soft BC $\\lambda_b={lam_b:g}$")
ax.set_xlabel("wall-clock time (s)"); ax.set_ylabel("relative $L^2$ error")
ax.set_title("accuracy against cost"); ax.legend(fontsize=7)

ax = axes[1]
ax.plot(T_CIRCLE, res_hard["boundary"], color=GEO_TEAL, lw=1.5, label="hard")
for lam_b, col in [(1.0, GEO_RUST), (100.0, GEO_DARK)]:
    ax.plot(T_CIRCLE, soft[lam_b]["boundary"], color=col, lw=1.2, label=f"soft, $\\lambda_b={lam_b:g}$")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel(r"angle on $\partial\mathbb{D}$"); ax.set_ylabel(r"$u_\theta$ on the boundary (should be 0)")
ax.set_title("the boundary condition"); ax.legend(fontsize=7.5)
plt.tight_layout(); plt.show()

Read the left panel horizontally, as a price list. The finite-element curve passes
*below and to the left of* every PINN marker: the best PINN, about ten seconds of
training, is matched by a mesh of 52,000 nodes solved in about half a second — **more
than an order of magnitude faster**. And FEM's error falls predictably as $O(h^2)$ —
quadrupling the node count cuts it fourfold — while the PINN's depends on the
architecture, the optimiser and the seed.

That is the honest verdict Lecture 12 asked for: **for a standard forward problem in two
dimensions, PINNs are not the tool**. Their case rests on the settings of slide 3, where
the classical toolbox struggles: domains that are hard to mesh, dimensions where meshes
are impossible, inverse problems, and parametric families. Exercise 2(c) makes the
dimension argument concrete.

> **Exercise 2 — balance and dimension.**
> (a) Sweep $\lambda_b$ over $10^{-2},\dots,10^{4}$ and plot residual loss against
> boundary error. You are tracing a Pareto front; where is the knee?
>
> (b) Slide 11 advises inspecting the gradient of each loss term separately. During
> training, record $\lVert\nabla_\theta L_{\text{res}}\rVert$ and
> $\lVert\nabla_\theta L_{\text{bc}}\rVert$. Set $\lambda_b$ adaptively to balance them, and
> compare with your best fixed value.
>
> (c) **Where PINNs win.** Solve $-\Delta u = 2d$ on the unit ball in $\mathbb R^d$ with
> $u = 0$ on the sphere (exact solution $1 - \lVert x\rVert^2$) for $d = 2, 5, 10, 20$. How
> does the PINN's cost grow with $d$? Estimate the node count a P1 mesh with $h = 0.05$
> would need in $d = 10$.

---
## 3. A PDE on a surface: Laplace–Beltrami by autodiff

Now the geometric case (slide 8). On the unit sphere $S^2$ solve

$$-\Delta_{S^2} u = f .$$

Two new features, both geometric.

**Differentiating along the surface.** We use an *embedded* architecture: the network
takes ambient coordinates $x\in\mathbb R^3$ and is restricted to the sphere. Its
Laplace–Beltrami operator can then be read off its ambient derivatives. Writing the
Euclidean Laplacian in polar form, $\Delta_{\mathbb R^3} = \partial_r^2 +
\frac{2}{r}\partial_r + \frac{1}{r^2}\Delta_{S^2}$, and setting $r = 1$ gives, for **any**
smooth extension $u$,

$$\Delta_{S^2} u \;=\; \underbrace{\operatorname{tr} H - x^\top H x}_{\text{tangential Hessian } P{:}H}
\;-\; \underbrace{2\, x\cdot\nabla u}_{\text{curvature term}},
\qquad H = \nabla^2 u,\quad P = I - xx^\top .$$

The lecture's notes warn that the tangential Hessian **alone** is not the right operator:
differentiating the projector produces the mean-curvature term. We will check how wrong
dropping it is.

**A closed surface.** $S^2$ has no boundary, and constants satisfy $\Delta_{S^2}c = 0$, so
the solution is only defined up to a constant — the kernel of $\Delta$ is the locally
constant functions, one per connected component (Tutorial 7's $b_0$). Two consequences:
$f$ must integrate to zero, and we fix the constant by asking $u$ to have mean zero.

In [ ]:
def laplace_beltrami(u, x, curvature_term=True):
    '''Laplace-Beltrami on the unit sphere of an ambient function u, by autodiff.'''
    g = grad(u.sum(), x, create_graph=True)[0]
    if g.requires_grad:
        H = torch.stack([grad(g[:, k].sum(), x, create_graph=True)[0] for k in range(3)], dim=1)
    else:                                          # u linear in x: Hessian identically zero
        H = torch.zeros(len(x), 3, 3)
    tangential = H.diagonal(dim1=1, dim2=2).sum(1) - torch.einsum("ni,nij,nj->n", x, H, x)
    return tangential - 2 * (x * g).sum(1) if curvature_term else tangential


def sphere_points(n, rng):
    v = rng.normal(size=(n, 3))
    return v / np.linalg.norm(v, axis=1, keepdims=True)


xs = torch.tensor(sphere_points(1000, rng), requires_grad=True)
X, Y, Z = xs[:, 0], xs[:, 1], xs[:, 2]
print("check on spherical harmonics, which satisfy  Lap Y = -l(l+1) Y :")
print(f"{'':18s} {'with curvature term':>22s} {'tangential Hessian only':>26s}")
for name, u, ell in [("z", Z, 1), ("xy", X * Y, 2), ("x^3 - 3xy^2", X**3 - 3 * X * Y**2, 3)]:
    good = (laplace_beltrami(u, xs) + ell * (ell + 1) * u).abs().max().item()
    bad = (laplace_beltrami(u, xs, curvature_term=False) + ell * (ell + 1) * u).abs().max().item()
    print(f"  {name:16s} {good:22.1e} {bad:26.2f}")

With the curvature term the operator is exact on every harmonic. Without it the error is
of order one, and the reason is clean: a degree-$\ell$ harmonic is homogeneous, so Euler's
theorem gives $x\cdot\nabla u = \ell\,u$. Dropping $-2\,x\cdot\nabla u$ therefore reports
the eigenvalue $\ell(\ell-1)$ instead of $\ell(\ell+1)$ — and for $\ell = 1$ it reports
$0$. A PINN trained with that operator would solve a different equation, converge happily,
and be wrong everywhere (Exercise 3a).

**The manufactured problem.** Take $u^\ast = e^{z} + xy - \sinh 1$, which is smooth, not a
finite sum of harmonics, and not rotationally symmetric. Using $\Delta_{S^2}g(z) =
(1-z^2)g'' - 2zg'$ for functions of $z$ alone, and $\Delta_{S^2}(xy) = -6xy$,

$$f \;=\; -\Delta_{S^2}u^\ast \;=\; (z^2 + 2z - 1)\,e^{z} + 6xy .$$

Both $u^\ast$ and $f$ have mean zero exactly — $(z^2+2z-1)e^z$ is the derivative of
$(z^2-1)e^z$, which vanishes at both poles — and the cell checks this, and the formula,
numerically.

In [ ]:
def u_exact_s(x):
    return torch.exp(x[:, 2]) + x[:, 0] * x[:, 1] - np.sinh(1.0)


def forcing_s(x):
    return (x[:, 2]**2 + 2 * x[:, 2] - 1) * torch.exp(x[:, 2]) + 6 * x[:, 0] * x[:, 1]


def fibonacci_sphere(n):
    i = np.arange(n)
    z = 1 - 2 * (i + 0.5) / n
    r = np.sqrt(1 - z * z)
    th = 2 * np.pi * i / ((1 + 5**0.5) / 2)
    return np.stack([r * np.cos(th), r * np.sin(th), z], axis=1)


chk = (-laplace_beltrami(u_exact_s(xs), xs) - forcing_s(xs)).abs().max().item()
xq = torch.tensor(fibonacci_sphere(400_000))              # a good quadrature rule on S^2
print(f"max | -Lap_S u* - f | = {chk:.1e}")
print(f"mean of u* on S^2 = {u_exact_s(xq).mean().item():.1e},   mean of f = {forcing_s(xq).mean().item():.1e}")

torch.manual_seed(0)
net_s = mlp(3)
u_s = lambda x: net_s(x).squeeze(1)

def loss_sphere(x):
    u = u_s(x)
    return ((-laplace_beltrami(u, x) - forcing_s(x))**2).mean() + u.mean()**2   # residual + gauge

XS_TEST = torch.tensor(sphere_points(20000, np.random.default_rng(1)))


def rel_l2_sphere(u_fn):
    with torch.no_grad():
        a, b = u_fn(XS_TEST), u_exact_s(XS_TEST)
        a, b = a - a.mean(), b - b.mean()                                    # compare up to the constant
        return (torch.linalg.norm(a - b) / torch.linalg.norm(b)).item()


res_s = train(loss_sphere, list(net_s.parameters()),
              sample=lambda r: (sphere_points(500, r),),
              fixed=(sphere_points(3000, np.random.default_rng(7)),),
              evaluate=lambda: rel_l2_sphere(u_s))
print(f"\nsphere PINN: Adam {res_s['t_adam']:.1f}s -> {res_s['err_adam']:.2e};  "
      f"+ L-BFGS {res_s['t_lbfgs']:.1f}s -> {res_s['err']:.2e}")

### The classical baseline on a surface

The finite-element method transfers to surfaces almost unchanged. Triangulate the sphere
— the convex hull of a Fibonacci lattice, exactly as in Tutorial 2 §2 — and assemble the
same P1 matrices from the same local formula with 3-D edge vectors. This is the
**cotangent Laplacian** of discrete differential geometry; the local formula
$e_a\cdot e_b/(4A)$ is its cotangent weights in disguise.

The closed surface needs the same gauge fixing as the PINN. We impose $\int u = 0$ with a
Lagrange multiplier, i.e. solve the bordered system
$\begin{pmatrix} K & m \\ m^\top & 0\end{pmatrix}\begin{pmatrix} u \\ \mu\end{pmatrix} =
\begin{pmatrix} Mf \\ 0 \end{pmatrix}$ with $m = M\mathbf 1$.

In [ ]:
def fem_sphere(n):
    pts = fibonacci_sphere(n)
    K, M = p1_matrices(pts, ConvexHull(pts).simplices)
    b = M @ forcing_s(torch.tensor(pts)).numpy()
    m = M @ np.ones(n)
    bordered = sp.bmat([[K, sp.csr_matrix(m[:, None])], [sp.csr_matrix(m[None, :]), None]]).tocsc()
    return pts, M, spla.spsolve(bordered, np.r_[b, 0.0])[:-1]


fem_s = []
print(f"{'nodes':>7s} {'rel. L2 error':>14s} {'time':>10s}")
for n in (200, 800, 3200, 12800, 51200):
    (pts, M, u_h), t_fem = best_time(fem_sphere, n)
    u_star = u_exact_s(torch.tensor(pts)).numpy()
    e = u_h - u_star
    err = np.sqrt(e @ M @ e / (u_star @ M @ u_star))
    fem_s.append((n, err, t_fem))
    print(f"{n:7d} {err:14.2e} {t_fem * 1000:8.1f}ms")
fem_s = np.array(fem_s)

xv = fibonacci_sphere(6000)
with torch.no_grad():
    uv = u_s(torch.tensor(xv)).numpy()
    us = u_exact_s(torch.tensor(xv)).numpy()
ev = np.abs((uv - uv.mean()) - (us - us.mean()))

fig = plt.figure(figsize=(12.0, 3.8))
for k, (vals, title, cmap) in enumerate([(us, "exact solution $u^*$ on $S^2$", "viridis"),
                                         (np.log10(ev + 1e-12), r"PINN error, $\log_{10}|u_\theta - u^*|$, mean-centred", "magma")]):
    ax = fig.add_subplot(1, 3, k + 1, projection="3d")
    sc = ax.scatter(*xv.T, c=vals, cmap=cmap, s=9, lw=0, vmin=-7 if k else None)
    ax.set_box_aspect((1, 1, 1), zoom=1.4); ax.view_init(elev=20, azim=35); ax.set_axis_off()
    ax.set_title(title); fig.colorbar(sc, ax=ax, shrink=0.6)

ax = fig.add_subplot(1, 3, 3)
ax.loglog(fem_s[:, 2], fem_s[:, 1], "o-", color=GEO_DARK, label="cotangent FEM")
ax.loglog(res_s["t_adam"] + res_s["t_lbfgs"], res_s["err"], "*", color=GEO_TEAL, ms=12, label="PINN")
ax.set_xlabel("wall-clock time (s)"); ax.set_ylabel("relative $L^2$ error")
ax.set_title("on the sphere"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The pattern of §2 repeats on the surface, with the same verdict: the cotangent FEM
converges at $O(h^2)$, reaches $10^{-4}$ in under a second, and continuing its line to the
PINN's accuracy costs a few seconds against the PINN's fourteen. What changed
between the disk and the sphere is instructive, though: for the PINN, *almost nothing*.
The architecture, the training loop and the error measurement are identical; only the
differential operator was swapped, and autodiff differentiated through the geometry for
us. That flexibility — not speed — is what a PINN offers, and it is what makes it the
natural starting point for the geometric problems of slide 9, where no mesh-based solver
is sitting ready.

> **Exercise 3 — operators on surfaces.**
> (a) Train the PINN with `curvature_term=False`. It will converge. Measure its error
> against $u^\ast$, and identify which equation it actually solved.
>
> (b) **A second route.** Feed the network $x/\lVert x\rVert$ instead of $x$. The extension
> is then constant along rays, so $\partial_r u = \partial_r^2 u = 0$ and the *plain*
> Euclidean Laplacian equals $\Delta_{S^2}$ on the sphere. Verify this numerically and
> compare the cost of the two approaches.
>
> (c) **Eigenfunctions** (slide 8). Find the first non-constant eigenfunction of
> $-\Delta_{S^2}$ with a PINN: learn $u_\theta$ and a scalar $\lambda$, with a residual loss
> plus penalties enforcing $\int u = 0$ and $\int u^2 = 1$. You should find $\lambda = 2$.
> What goes wrong without the normalisation?

---
## 4. Deep Ritz: minimising an energy instead of a residual

The Poisson problem is the Euler–Lagrange equation of a variational principle (slide 9):
among functions vanishing on the boundary, $u^\ast$ minimises the **Dirichlet energy**

$$E[u] \;=\; \int_{\mathbb D}\Big(\tfrac12\lVert\nabla u\rVert^2 - f\,u\Big)\,dx ,$$

the same energy Tutorial 3 §4 used as a regulariser. **Deep Ritz** minimises $E[u_\theta]$
directly. It needs only *first* derivatives, so each step is cheaper than a residual step.
And the energy integral must be estimated from points: we try two point sets of the same
size, random and a quasi-uniform sunflower lattice.

For this problem the minimum value is available in closed form,
$E[u^\ast] = -\tfrac12\int f u^\ast = -4\pi/3$.

In [ ]:
def sunflower_disk(n):
    '''A quasi-uniform lattice of n points in the unit disk (equal weights).'''
    j = np.arange(n)
    r = np.sqrt((j + 0.5) / n)
    t = j * np.pi * (3 - np.sqrt(5))
    return np.stack([r * np.cos(t), r * np.sin(t)], axis=1)


def energy(u_fn, x):
    u = u_fn(x)
    gu = grad(u.sum(), x, create_graph=True)[0]
    return np.pi * (0.5 * (gu**2).sum(1) - forcing(x) * u).mean()      # area(D) * mean


E_EXACT = -4 * np.pi / 3
X_QUAD = torch.tensor(sunflower_disk(200_000), requires_grad=True)       # accurate quadrature, for evaluation
E_STAR = energy(u_exact, X_QUAD).item()
print(f"quadrature check:  E[u*] = {E_STAR:.6f}   (exact {E_EXACT:.6f})")

ritz = {}
for label, fixed in [("random points", disk_points(4000, np.random.default_rng(7))),
                     ("sunflower lattice", sunflower_disk(4000))]:
    torch.manual_seed(0)
    net = mlp(2)
    u_r = lambda x, net=net: (1 - (x**2).sum(1)) * net(x).squeeze(1)
    out = train(lambda x, u=u_r: energy(u, x), list(net.parameters()),
                sample=lambda r: (disk_points(500, r),), fixed=(fixed,),
                evaluate=lambda u=u_r: rel_l2(u))
    out["E"] = energy(u_r, X_QUAD).item()
    diff = lambda x, u=u_r: u(x) - u_exact(x)
    gd = grad(diff(X_QUAD).sum(), X_QUAD, create_graph=False)[0]
    out["half_H1"] = (np.pi * 0.5 * (gd**2).sum(1).mean()).item()
    out["u"] = u_r
    ritz[label] = out

print(f"\n{'':32s} {'time':>7s} {'rel. L2 error':>14s} {'E[u] - E[u*]':>14s} {'1/2 |u-u*|^2_H1':>17s}")
print(f"{'residual PINN (§1)':32s} {res_hard['t_adam'] + res_hard['t_lbfgs']:6.1f}s {res_hard['err']:14.2e}")
for label, o in ritz.items():
    print(f"{'Deep Ritz, ' + label:32s} {o['t_adam'] + o['t_lbfgs']:6.1f}s {o['err']:14.2e} "
          f"{o['E'] - E_STAR:14.2e} {o['half_H1']:17.2e}")

Three results, in increasing order of importance.

**Deep Ritz is cheaper per step**, as slide 9 promised — first derivatives only.

**Its accuracy depends on the quadrature, and dramatically so.** On random points it is
far worse than the residual PINN; on a quasi-uniform lattice of the *same size* it
improves by well over an order of magnitude. This is not a tuning accident but a
structural difference between the two losses:

- the **residual** loss is a sum of squares, zero at every point for the exact solution.
  So $u^\ast$ minimises it for *any* choice of points; sampling only affects how well the
  network is pinned down *between* them.
- the **energy** is an integral, and a point set defines a *different* functional. L-BFGS
  minimises that functional very well — and its minimiser is not $u^\ast$. The quadrature
  error becomes solution error. Tutorial 3 §1's lesson, that quadrature beats sampling,
  here decides the answer.

**The energy certifies the solution.** For any $u$ vanishing on the boundary, integrating
by parts against the weak form of the PDE gives the exact identity

$$E[u] - E[u^\ast] \;=\; \tfrac12\int_{\mathbb D}\lVert\nabla(u - u^\ast)\rVert^2\,dx \;\ge\; 0 ,$$

which the last two columns confirm. Compare slide 11: a *low residual* does not certify a
correct solution. A *lower energy*, for a variational problem, does — it strictly ranks
any two candidates by their distance to the truth in the energy norm, **without knowing the
truth**. That is the practical reason to look for a variational principle before reaching
for a residual.

> **Exercise 4 — energies, and learning what is unknown.**
> (a) Use $E[u]$ alone to rank the four disk solutions of §1–§2 (hard, soft
> $\lambda_b = 1$, soft $\lambda_b = 100$, and your Exercise 1 variants). Does the ranking
> agree with the true errors? Why must the soft-constraint solutions be treated with care
> here?
>
> (b) Train Deep Ritz with Adam on freshly resampled random points only, without the
> L-BFGS finish. Is it as badly biased as the fixed random set? Explain in terms of what
> each optimiser is minimising.
>
> (c) **An inverse problem** (slide 10; a good mini-project seed). Take
> $-\nabla\cdot(a\nabla u) = f$ on the disk with the true conductivity
> $a^\ast(x) = 1 + \tfrac12 x_1$ and the $u^\ast$ above, and compute $f$ by autodiff.
> Give yourself 40 noisy observations of $u^\ast$, and learn $u_\theta$ and $a_\phi > 0$
> jointly from the residual plus a data term. Where in the disk is $a$ recovered well, and
> where badly? Relate the answer to where $\nabla u^\ast$ vanishes — the equation cannot
> see $a$ where the flux $a\nabla u$ does not depend on it.

---
## 5. What to take away

- **A PINN is a network used as a trial function**, trained so the PDE holds at sampled
  points. Autodiff gives its derivatives exactly; the errors come from approximation and
  sampling, and must be measured against something other than the training loss.
- **Build boundary conditions into the ansatz when you can.** A hard constraint is exact
  and spends the network's capacity on the PDE; a soft penalty makes the two losses
  compete, and no single weight removes the trade-off.
- **Benchmark against a classical solver, and report it honestly.** On the disk and on the
  sphere, finite elements reach the same accuracy several to twenty times faster, with a
  predictable convergence rate. PINNs earn their place elsewhere: hard-to-mesh
  domains, high dimension, inverse and parametric problems.
- **Geometry enters through the operator.** On a surface only the differential operator
  changes, and autodiff differentiates through the embedding — provided the operator is
  right: dropping the curvature term turns $\ell(\ell+1)$ into $\ell(\ell-1)$.
- **Energies and residuals behave differently.** A residual loss is minimised by the true
  solution on any point set; an energy loss inherits its quadrature error. But a lower
  energy certifies a better solution, which no residual can do.

### Next

**Lecture 13** adds time and nonlinearity — heat-type and curvature flows — and turns
the failure modes previewed on slide 11 into remedies, before introducing neural
operators. **Tutorial 13** solves a time-dependent geometric PDE and meets those
failures directly.

### Further reading

- Raissi, Perdikaris & Karniadakis, "Physics-informed neural networks", *J. Comput. Phys.* **378** (2019) — the framework of §1.
- E & Yu, "The Deep Ritz method", *Commun. Math. Stat.* **6** (2018) — §4.
- Lu, Pestourie, Yao, Wang, Verdugo & Johnson, "Physics-informed neural networks with hard constraints for inverse design", *SIAM J. Sci. Comput.* **43** (2021) — hard constraints, §2.
- Wang, Teng & Perdikaris, "Understanding and mitigating gradient flow pathologies in physics-informed neural networks", *SIAM J. Sci. Comput.* **43** (2021) — the loss balancing of Exercise 2(b).
- Crane, de Goes, Desbrun & Schröder, "Digital geometry processing with discrete exterior calculus", SIGGRAPH course (2013) — the cotangent Laplacian of §3.